# v5 Pilot Diagnostics

Loads `RESULTS/pilot_report*.json` and visualizes:
- Per-cell Gate 2 retention rate vs floor
- Chain length distributions per cell
- Event type distributions per cell

Use after `python -m v5.run_pilot` (mock or real).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPORT_PATH = Path('../RESULTS/pilot_report_mock.json')  # change to real path when available
with open(REPORT_PATH) as f:
    report = json.load(f)

cells_df = pd.DataFrame(report['cells'])
cells_df[['cell', 'n_streams', 'n_events_total', 'n_chains_post_gate2', 'retention_rate', 'gate2_pass']]

In [ ]:
# Per-cell retention rate vs Gate 2 floor
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(cells_df['cell'], cells_df['retention_rate'], color=['green' if p else 'red' for p in cells_df['gate2_pass']])
ax.axhline(0.50, color='black', linestyle='--', label='Gate 2 floor (50%)')
ax.set_ylabel('Retention rate')
ax.set_title('Gate 2 Retention per Cell')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Chain length stats per cell
stats_df = cells_df[['cell', 'chain_length_mean', 'chain_length_min', 'chain_length_max', 'chain_length_median']]
stats_df

In [ ]:
# Event type distribution heatmap
import numpy as np

all_etypes = set()
for c in report['cells']:
    all_etypes.update(c['event_type_distribution'].keys())
all_etypes = sorted(all_etypes)

matrix = np.zeros((len(report['cells']), len(all_etypes)))
for i, c in enumerate(report['cells']):
    total = sum(c['event_type_distribution'].values())
    for j, et in enumerate(all_etypes):
        matrix[i, j] = c['event_type_distribution'].get(et, 0) / total if total else 0

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(matrix, aspect='auto', cmap='YlGnBu')
ax.set_xticks(range(len(all_etypes)))
ax.set_xticklabels(all_etypes, rotation=90)
ax.set_yticks(range(len(report['cells'])))
ax.set_yticklabels([c['cell'] for c in report['cells']])
ax.set_title('Event type fraction per cell')
plt.colorbar(im)
plt.tight_layout()
plt.show()